<a href="https://colab.research.google.com/github/abdullahawan0043-glitch/Flyrank-machine-learning-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
"""
Finding 1 - "The Anatomy of Growing Content" (Finding #1, CONFIRMED)

The paper compares growing vs. declining pages and reports growing content
is 37.6% longer and 20% younger, calling this a "directionally robust"
observational pattern.

My methodology question: Where does the "up"/"down" trend label come from
relative to when word_count and age were measured? The paper defines trend
from a 30-day-vs-previous-30-day impression change, but word count and age
are measured at the time of the snapshot, not at the start of that trend
window. If a page's word count was increased DURING the growth window
(e.g., because someone refreshed it, which is exactly what the paper later
recommends doing), then word count could be partly a CONSEQUENCE of early
growth rather than a cause of it. The paper is careful to call this
"directionally robust" and "observational," which is the right instinct -
I would only ask that the timing of the word-count snapshot relative to
the trend window be stated explicitly, so a reader can judge whether this
is a leading indicator or a trailing one.

Finding 2 - "What Predicts Health?" (ML Appendix, Random Forest feature importance)

The paper reports Average Position as the top predictor of Health Score
(43% importance), and states this is "descriptive rather than causal"
because Health Score is partly constructed from these same inputs.

My methodology question: Health Score is explicitly defined earlier in the
paper as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll
Depth (20 pts) - meaning Average Position is a direct arithmetic
ingredient of the label the model is predicting. This is a textbook
label-derived feature case. The paper's own caveat ("importance is
descriptive rather than causal") is honest and appropriate, but I would
ask: given that ~73% of the importance (Position 43% + Impressions 32%)
comes from features that are literally summed into the label, does the
remaining ranking (Scroll Depth, CTR) still mean anything on its own, or
would the appendix be clearer if it reported feature importance only on
the NON-constituent features (e.g., word count, content age, AI sessions)
to show what predicts health beyond its own ingredients?
"""

'\nFinding 1 - "The Anatomy of Growing Content" (Finding #1, CONFIRMED)\n\nThe paper compares growing vs. declining pages and reports growing content\nis 37.6% longer and 20% younger, calling this a "directionally robust"\nobservational pattern.\n\nMy methodology question: Where does the "up"/"down" trend label come from\nrelative to when word_count and age were measured? The paper defines trend\nfrom a 30-day-vs-previous-30-day impression change, but word count and age\nare measured at the time of the snapshot, not at the start of that trend\nwindow. If a page\'s word count was increased DURING the growth window\n(e.g., because someone refreshed it, which is exactly what the paper later\nrecommends doing), then word count could be partly a CONSEQUENCE of early\ngrowth rather than a cause of it. The paper is careful to call this\n"directionally robust" and "observational," which is the right instinct -\nI would only ask that the timing of the word-count snapshot relative to\nthe trend 

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
import os, subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/abdullahawan0043-glitch/Flyrank-machine-learning-internship"
REPO_DIR = "Flyrank-machine-learning-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "sessions_90d", "engagement_rate"]

def prep(d):
    X = d[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = (d["trend_direction"] == "down").astype(int)
    return X, y

X_all, y_all = prep(df)
base_rate = y_all.mean()
print(f"Base rate (share declining): {base_rate:.3f}")

Base rate (share declining): 0.542


In [ ]:
"""
Before/after: random split (dishonest) vs. grouped split (honest, by client_id)

My Week 5 model already used a grouped split. Here I deliberately ALSO
build a random row-split version to show the gap the leakage-hunting
skill describes - the difference between the two numbers is itself the
finding.
"""

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def train_eval(train_df, test_df, label):
    X_train, y_train = prep(train_df)
    X_test, y_test = prep(test_df)
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_SEED, class_weight="balanced")
    rf.fit(X_train, y_train)
    scores = rf.predict_proba(X_test)[:, 1]
    p50 = precision_at_k(scores, y_test, 50)
    auc = roc_auc_score(y_test, scores)
    print(f"{label}: Precision@50={p50:.3f}, ROC-AUC={auc:.3f}")
    return p50, auc

# BEFORE: random row split (no group protection)
train_r, test_r = train_test_split(df, test_size=0.3, random_state=RANDOM_SEED)
p50_random, auc_random = train_eval(train_r, test_r, "BEFORE - random split")

# AFTER: grouped split by client_id (honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_g, test_g = df.iloc[train_idx], df.iloc[test_idx]
p50_grouped, auc_grouped = train_eval(train_g, test_g, "AFTER  - grouped split (by client_id)")

print(f"\nGap: Precision@50 dropped by {p50_random - p50_grouped:.3f} when moving to the honest split.")
print("This gap is the finding: it shows how much of the random-split score was client memorization, not real signal.")

BEFORE - random split: Precision@50=0.940, ROC-AUC=0.748
AFTER  - grouped split (by client_id): Precision@50=0.540, ROC-AUC=0.603

Gap: Precision@50 dropped by 0.400 when moving to the honest split.
This gap is the finding: it shows how much of the random-split score was client memorization, not real signal.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
assert "trend_direction" not in features and "trend_pct" not in features, "Label-derived column found in features!"
print("Confirmed: no label-derived columns in the feature list.")

X_train, y_train = prep(train_g)
X_test, y_test = prep(test_g)

# Retrain the honest model here so it's available in this cell
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_SEED, class_weight="balanced")
rf.fit(X_train, y_train)

# Deliberate leak test: add a label-derived column and watch the score jump
X_train_leaky = X_train.copy()
X_test_leaky = X_test.copy()
X_train_leaky["leaky_trend_pct"] = train_g["trend_pct"].values
X_test_leaky["leaky_trend_pct"] = test_g["trend_pct"].values

rf_leaky = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_SEED, class_weight="balanced")
rf_leaky.fit(X_train_leaky, y_train)
leaky_scores = rf_leaky.predict_proba(X_test_leaky)[:, 1]
leaky_p50 = precision_at_k(leaky_scores, y_test, 50)
leaky_auc = roc_auc_score(y_test, leaky_scores)

print(f"\nHonest model (grouped split): Precision@50={p50_grouped:.3f}, ROC-AUC={auc_grouped:.3f}")
print(f"Deliberately leaky model (+trend_pct): Precision@50={leaky_p50:.3f}, ROC-AUC={leaky_auc:.3f}")
print("The jump toward a near-perfect score confirms the test harness correctly detects leakage when it's present.")

# Feature importance sanity check on the HONEST model
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("\nHonest model feature importances:")
print(importances)
print("\nTop feature is not suspiciously dominant (no single feature >90% importance) - consistent with real, distributed signal rather than a hidden leak.")

Confirmed: no label-derived columns in the feature list.

Honest model (grouped split): Precision@50=0.540, ROC-AUC=0.603
Deliberately leaky model (+trend_pct): Precision@50=1.000, ROC-AUC=1.000
The jump toward a near-perfect score confirms the test harness correctly detects leakage when it's present.

Honest model feature importances:
impressions_90d           0.292545
avg_position              0.218726
content_age_days          0.178681
word_count                0.131147
days_since_last_update    0.063023
sessions_90d              0.052496
ctr                       0.050214
engagement_rate           0.013169
dtype: float64

Top feature is not suspiciously dominant (no single feature >90% importance) - consistent with real, distributed signal rather than a hidden leak.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
"""
Rewriting my own earlier claims into public-safe language:

ORIGINAL (Week 5 notebook): "Random Forest gave the strongest lift over
the baseline."
REWRITE: "In this observed test split, Random Forest showed a directional
improvement in Precision@50 over the Week 4 baseline rule; this is a
decision-support signal for prioritizing review, not a guarantee that any
individual flagged page is truly declining."

ORIGINAL (Week 4 notebook): "The learned model finds roughly 3x more true
declining pages in the top 50 than a fixed rule."
REWRITE: "On this measured dataset and split, the learned model's top-50
ranked pages contained roughly 3x as many observed decliners as the fixed
rule's top 50 - a directional result specific to this data slice, not a
claim that generalizes to all FlyRank clients without re-validation."

ORIGINAL (this week, about the random-split number): "The model works
well."
REWRITE: "The model's random-split score was measured to be inflated
relative to its grouped-split score; the honest, decision-support number
is the grouped-split result, since it reflects performance on clients the
model has not seen before."
"""

'\nRewriting my own earlier claims into public-safe language:\n\nORIGINAL (Week 5 notebook): "Random Forest gave the strongest lift over\nthe baseline."\nREWRITE: "In this observed test split, Random Forest showed a directional\nimprovement in Precision@50 over the Week 4 baseline rule; this is a\ndecision-support signal for prioritizing review, not a guarantee that any\nindividual flagged page is truly declining."\n\nORIGINAL (Week 4 notebook): "The learned model finds roughly 3x more true\ndeclining pages in the top 50 than a fixed rule."\nREWRITE: "On this measured dataset and split, the learned model\'s top-50\nranked pages contained roughly 3x as many observed decliners as the fixed\nrule\'s top 50 - a directional result specific to this data slice, not a\nclaim that generalizes to all FlyRank clients without re-validation."\n\nORIGINAL (this week, about the random-split number): "The model works\nwell."\nREWRITE: "The model\'s random-split score was measured to be inflated\nrelat

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.